# ORTHRUS-MSTC-PIDS — PAI-DSW AllInOne Notebook

本 Notebook 是 **ORTHRUS-MSTC-PIDS** 项目在阿里云 PAI-DSW 上的 AllInOne
入口。它与既有 Google Colab 版本
`notebooks/ORTHRUS_MSTC_PIDS_AllInOne_Colab.ipynb` 语义一致，但将：

* Google Drive → `/mnt/workspace/mstc_pids`
* Colab clone → 本地 GitHub ZIP 安全解压
* Colab `/content/orthrus` → `/mnt/workspace/.../<repo-top>`
* PostgreSQL 自动安装 → 已存在的 artifacts 直接复用 detection_only

> **重要使用约定**
>
> 1. 数据目录已经在 PAI-DSW 中准备：
>    `/mnt/workspace/mstc_pids`（即原 Google Drive `mstc_pids` 的完整副本）。
> 2. 源码以 GitHub branch ZIP 形式上传：
>    `/mnt/workspace/orthrus-fix-c8-loader-telemetry-persist.zip`。
> 3. **所有重任务默认都是关闭的**（preprocessing / smoke / main matrix /
>    ablations / resume）。
> 4. **不要直接 Run All 后离开**。先按顺序运行 0–9 单元格完成 setup，
>    再按需打开 smoke → main matrix。
> 5. Smoke 通过后再设置 `RUN_MAIN_MATRIX=True`，然后运行 Main Matrix 单元格。
> 6. 当 build_graphs / metadata / embed_nodes / embed_edges 全部完整时，
>    完全不需要 PostgreSQL，pipeline 自动以 `detection_only` 模式运行。
> 7. 本 Notebook **不会**删除或破坏 `/mnt/workspace` 下任何已有文件；
>    解压源码时使用 staging 目录，验证成功后才会切换。
> 8. 既有 Google Colab Notebook 仍然保留，本 Notebook 是独立新增文件，
>    不修改、不替换原 Colab Notebook。


## 0. Unified Parameters

In [ ]:
from pathlib import Path

WORKSPACE_ROOT = Path("/mnt/workspace")
MSTC_ROOT = WORKSPACE_ROOT / "mstc_pids"
ARTIFACT_ROOT = MSTC_ROOT / "artifacts"
DATA_ROOT = MSTC_ROOT / "data"

SOURCE_ARCHIVE = (
    WORKSPACE_ROOT /
    "orthrus-fix-c8-loader-telemetry-persist.zip"
)
SOURCE_KIND = "github_zip"

REPOSITORY_REF = "fix/c8-loader-telemetry-persist"

# Pin to the production fix commit.
# In a ZIP source without .git, git_commit cannot be verified;
# the source archive SHA256 + this constant serve as the
# "is this the right ZIP?" cross-check (see Section 2).
EXPECTED_PRODUCTION_COMMIT = (
    "d55a075d752e02d06f87189901d2221fedd4fd4b"
)

DATASET = "THEIA_E3"
SEEDS = [0]
CORPUS_SCOPE = "train_only"

# ----- Source preparation / extraction --------------------------------
EXTRACT_SOURCE = True
FORCE_EXTRACT_SOURCE = False

# ----- Dependencies ---------------------------------------------------
INSTALL_DEPENDENCIES = True

# ----- Database ------------------------------------------------------
ALLOW_NETWORK_FETCH_GROUND_TRUTH = True
RUN_DB_PREFLIGHT = False
ORTHRUS_DB_HOST = ""
ORTHRUS_DB_PORT = ""
ORTHRUS_DB_USER = ""
ORTHRUS_DB_PASSWORD = ""

# ----- Preprocessing -------------------------------------------------
RUN_PREPROCESS_BENCHMARK = False
RUN_PREPROCESS = False
FORCE_PREPROCESS = False
PREPROCESS_SUBSTAGES = (
    "build_graphs,embed_nodes,embed_edges"
)

# ----- Smoke / Main / Ablations / Resume ------------------------------
RUN_BASELINE_SMOKE = False
SMOKE_MAX_WINDOWS_PER_SPLIT = 2

RUN_MAIN_MATRIX = False
FORCE_MAIN_MATRIX = False

RUN_ABLATIONS = False
FORCE_ABLATIONS = False

RUN_MANUAL_RESUME = False
EXPERIMENT_GROUP = "ablation"
RESUME_CONFIG_REL = "config/experiments/mstc_full.yml"
CHECKPOINT = Path("")

# ----- Result collection / display -----------------------------------
RUN_COLLECT_EXPORT = False
RUN_DISPLAY_RESULTS = False

assert DATASET in {"THEIA_E3", "THEIA_E5"}
assert CORPUS_SCOPE in {"train_only", "official_full_dataset"}

print(f"Dataset       = {DATASET}")
print(f"corpus_scope  = {CORPUS_SCOPE}")
print(f"ref           = {REPOSITORY_REF}")
print(f"Expected commit (cross-check only): {EXPECTED_PRODUCTION_COMMIT}")
print(f"Source archive: {SOURCE_ARCHIVE}")
print("All heavy-stage switches are disabled by default.")
PAI_MSTC_ROOT = "/mnt/workspace/mstc_pids"
PAI_SOURCE_ARCHIVE = "/mnt/workspace/orthrus-fix-c8-loader-telemetry-persist.zip"
assert str(MSTC_ROOT) == PAI_MSTC_ROOT
assert str(SOURCE_ARCHIVE) == PAI_SOURCE_ARCHIVE


## 1. PAI-DSW Workspace and Source Preparation

In [ ]:
import os
import sys
import shutil
import zipfile
from pathlib import Path

ENVIRONMENT_DIR = ARTIFACT_ROOT / "environment"
ENVIRONMENT_DIR.mkdir(parents=True, exist_ok=True)

WORKSPACE_ROOT.mkdir(parents=True, exist_ok=True)
MSTC_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
os.environ["ORTHRUS_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["ORTHRUS_DATA_ROOT"] = str(DATA_ROOT)

print(f"Workspace root:   {WORKSPACE_ROOT}")
print(f"MSTC root:        {MSTC_ROOT}")
print(f"Artifact root:    {ARTIFACT_ROOT}")
print(f"Data root:        {DATA_ROOT}")
print(f"Environment log:  {ENVIRONMENT_DIR}")


## 2. Source Verification and Ground Truth Preparation

This section verifies that the uploaded source archive matches the
expected production ref, then extracts it into a staging directory
before swapping it in. The previous Colab Notebook
(fixed under `fix/c8-loader-telemetry-persist`) is cross-checked by
string match. No `rm -rf` is used against existing source.

In [ ]:
import hashlib
import shutil
import zipfile
from pathlib import Path


def sha256_file(path: Path) -> str:
    """Streamed SHA-256 for the source archive."""
    h = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()


REQUIRED_SOURCE_PATHS = [
    "src/experiments/run_experiment.py",
    "src/experiments/run_matrix.py",
    "src/run_metadata.py",
    "src/config.py",
    "config/orthrus.yml",
    "config/experiments/baseline.yml",
    "config/experiments/mstc_full.yml",
]


def verify_project_root(root: Path) -> None:
    """Hard-fail when the extracted tree is missing expected files."""
    missing = [p for p in REQUIRED_SOURCE_PATHS
               if not (root / p).is_file()]
    if missing:
        raise FileNotFoundError(
            "Source tree missing required files: " + ", ".join(missing)
        )


def _detect_zip_top_dir(archive: Path) -> str:
    """Pick the single top-level directory inside the ZIP.

    GitHub ZIP archives always wrap everything in one folder.
    Reject archives with multiple distinct top-level entries or
    with loose files at the root, so we never silently flatten the
    user-supplied archive.
    """
    with zipfile.ZipFile(archive, "r") as zf:
        names = zf.namelist()
    tops = sorted({Path(n).parts[0] for n in names if n.strip()})
    if len(tops) != 1:
        raise RuntimeError(
            "Source archive is expected to wrap a single top-level "
            "directory; got: " + ", ".join(tops)
        )
    return tops[0]


def prepare_source_archive(
    archive: Path,
    workspace: Path,
    *,
    force: bool = False,
) -> Path:
    """Extract the GitHub ZIP into a per-build directory.

    - Missing archive → FileNotFoundError.
    - Existing valid source tree + force=False → reuse.
    - force=True → extract into a staging dir, validate, then move.
    No `shutil.rmtree` against an existing project root.
    """
    if not archive.is_file():
        raise FileNotFoundError(f"Source archive not found: {archive}")

    archive_sha256 = sha256_file(archive)
    top_dir = _detect_zip_top_dir(archive)
    project_root = workspace / top_dir

    if project_root.is_dir() and verify_project_root(project_root) or False:
        pass

    if project_root.is_dir():
        if not force:
            print(
                f"Reusing existing source tree: {project_root}"
            )
            return project_root
        # If force=True we move the existing tree aside, never delete it.
        aside = workspace / f".{top_dir}.bak"
        n = 1
        while aside.exists():
            aside = workspace / f".{top_dir}.bak{n}"
            n += 1
        shutil.move(str(project_root), str(aside))
        print(
            f"force=True; moved existing tree aside: {aside}"
        )

    staging = workspace / (
        f".orthrus_pai_extract_staging_{archive_sha256[:12]}"
    )
    if staging.exists():
        shutil.rmtree(staging)
    staging.mkdir(parents=True)

    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(staging)

    extracted_root = staging / top_dir
    if not extracted_root.is_dir():
        raise RuntimeError(
            f"Staging extraction missing top directory: {extracted_root}"
        )
    verify_project_root(extracted_root)

    shutil.move(str(extracted_root), str(project_root))
    shutil.rmtree(staging)
    print(f"Extracted source tree to: {project_root}")
    return project_root


def cross_check_colab_notebook(project_root: Path) -> None:
    """Confirm the Colab Notebook inside the ZIP contains the right ref.

    Looks for both the branch ref and the pinned production commit
    inside the Colab Notebook JSON. This guards against uploading a
    wrong-ZIP accident.
    """
    colab_nb = project_root / "notebooks" / \
        "ORTHRUS_MSTC_PIDS_AllInOne_Colab.ipynb"
    if not colab_nb.is_file():
        return
    text = colab_nb.read_text(encoding="utf-8")
    if REPOSITORY_REF not in text:
        raise RuntimeError(
            f"Colab Notebook in the archive does not mention "
            f"{REPOSITORY_REF!r}."
        )
    if EXPECTED_PRODUCTION_COMMIT not in text:
        raise RuntimeError(
            f"Colab Notebook in the archive does not mention the "
            f"pinned commit {EXPECTED_PRODUCTION_COMMIT!r}."
        )
    print(f"Colab Notebook cross-check: {colab_nb} passed.")


Run the helpers. `SOURCE_KIND` is `"github_zip"`; the archive has no
`.git`, so `git_commit` will remain `null` in the recorded environment
(see Section 24 — that's the documented behaviour, not a forgery).

SOURCE_ARCHIVE_SHA256 is recorded for later cross-checks.

In [ ]:
if not EXTRACT_SOURCE:
    raise RuntimeError(
        "EXTRACT_SOURCE=False with no pre-existing PROJECT_ROOT. "
        "The PAI-DSW pipeline requires the GitHub ZIP to be extracted "
        "before source verification."
    )

PROJECT_ROOT = prepare_source_archive(
    SOURCE_ARCHIVE,
    WORKSPACE_ROOT,
    force=FORCE_EXTRACT_SOURCE,
)
verify_project_root(PROJECT_ROOT)
cross_check_colab_notebook(PROJECT_ROOT)

SOURCE_ARCHIVE_SHA256 = sha256_file(SOURCE_ARCHIVE)
print(f"Source archive SHA256: {SOURCE_ARCHIVE_SHA256}")

(ENVIRONMENT_DIR / "source_archive_sha256.txt").write_text(
    f"{SOURCE_ARCHIVE}\n{SOURCE_ARCHIVE_SHA256}\n",
    encoding="utf-8",
)

# Optional .git verification (only if a .git directory exists).
if (PROJECT_ROOT / ".git").is_dir():
    import subprocess
    head = subprocess.run(
        ["git", "-C", str(PROJECT_ROOT), "rev-parse", "HEAD"],
        check=False, capture_output=True, text=True,
    )
    if head.returncode == 0:
        print(f"Git commit inside PROJECT_ROOT: {head.stdout.strip()}")
    else:
        print("PROJECT_ROOT has .git but HEAD could not be resolved.")
else:
    print(
        "Source is a GitHub ZIP archive; Git commit cannot be "
        "verified from .git metadata."
    )

# Add src to sys.path.
SRC_ROOT = str(PROJECT_ROOT / "src")
if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)
print(f"sys.path updated: {SRC_ROOT}")


Ground Truth is a Git submodule inside the project. On PAI-DSW we
(A) reuse a fully-prepared tree, (B) fall back to a local ZIP, or
(C) `git clone` from `ProvenanceAnalytics/ground-truth` and
`checkout` the pinned commit. HEAD is never used in place of the pin.

In [ ]:
import shutil
import subprocess
import zipfile
from pathlib import Path

GROUND_TRUTH_ROOT = (
    PROJECT_ROOT / "Ground_Truth" / "darpa"
)
GROUND_TRUTH_REPO = (
    "https://github.com/ProvenanceAnalytics/ground-truth.git"
)
GROUND_TRUTH_COMMIT = (
    "012f321f46137650496e639b0ad7e0a66db07a73"
)
GROUND_TRUTH_ARCHIVE = (
    WORKSPACE_ROOT /
    f"ground-truth-{GROUND_TRUTH_COMMIT}.zip"
)


def _gt_already_initialised(root: Path) -> bool:
    """Heuristic: .git present, or any non-trivial data file present.

    Submodule data is committed via tags/releases rather than living
    on a single HEAD; we treat a populated tree as ready.
    """
    if not root.is_dir():
        return False
    if (root / ".git").is_file() or (root / ".git").is_dir():
        return True
    return any(p.is_file() for p in root.rglob("*") if p.is_file())


def _extract_archive(archive: Path, dest: Path) -> None:
    if dest.exists():
        shutil.move(
            str(dest),
            str(dest.with_name(dest.name + ".bak")),
        )
    dest.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archive, "r") as zf:
        zf.extractall(dest)


def ensure_ground_truth(
    *,
    root: Path,
    repo: str,
    commit: str,
    archive: Path,
    allow_network: bool,
    network_fetcher,
) -> Path:
    """Reuse / unzip / clone the ground-truth submodule data.

    `network_fetcher(root, repo, commit)` performs the clone+checkout
    step. It is injected so this helper stays IO-policy free.
    """
    if _gt_already_initialised(root):
        print(f"Ground Truth already initialised: {root}")
        return root

    if archive.is_file():
        print(f"Extracting Ground Truth archive: {archive}")
        _extract_archive(archive, root)
        if _gt_already_initialised(root):
            return root

    if not allow_network:
        raise RuntimeError(
            "Ground Truth is incomplete and ALLOW_NETWORK_FETCH_GROUND_TRUTH "
            "is False. Upload the matching archive or enable network fetch."
        )

    print(f"Cloning Ground Truth from {repo} at {commit}")
    network_fetcher(root, repo, commit)
    if not _gt_already_initialised(root):
        raise RuntimeError(
            f"Ground Truth clone failed to populate {root}."
        )
    return root


In [ ]:
import subprocess


def _git_clone_ground_truth(
    dest: Path, repo: str, commit: str,
) -> None:
    """Network fetch: clone then strictly checkout the pinned commit.

    We never use the remote default branch HEAD in place of `commit`.
    """
    dest.parent.mkdir(parents=True, exist_ok=True)
    if (dest / ".git").is_dir() or (dest / ".git").is_file():
        # Reuse a previous clone.
        subprocess.run(
            ["git", "-C", str(dest), "fetch", "--all"],
            check=False,
        )
    else:
        subprocess.run(
            ["git", "clone", repo, str(dest)],
            check=True,
        )
    subprocess.run(
        ["git", "-C", str(dest), "checkout", commit],
        check=True,
    )


In [ ]:
ensure_ground_truth(
    root=GROUND_TRUTH_ROOT,
    repo=GROUND_TRUTH_REPO,
    commit=GROUND_TRUTH_COMMIT,
    archive=GROUND_TRUTH_ARCHIVE,
    allow_network=ALLOW_NETWORK_FETCH_GROUND_TRUTH,
    network_fetcher=_git_clone_ground_truth,
)
print(f"Pinned Ground Truth commit: {GROUND_TRUTH_COMMIT}")
print(f"Ground Truth root: {GROUND_TRUTH_ROOT}")


## 3. Resource Preflight

Inspect the PAI-DSW runtime: Python, torch, CUDA, GPU, CPU RAM,
workspace disk. Do **not** upgrade/downgrade torch or CUDA; the
pipeline expects the installed versions to be honoured.

In [ ]:
import shutil
import subprocess
import sys
import psutil
import torch

print(f"Python:     {sys.version.split()[0]}")
print(f"Executable: {sys.executable}")
print(f"torch:      {torch.__version__}")
print(f"torch CUDA: {torch.version.cuda}")
print(f"CUDA avail: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:        {torch.cuda.get_device_name(0)}")
    if shutil.which("nvidia-smi"):
        subprocess.run(["nvidia-smi"], check=False)

try:
    import torch_geometric
    print(f"PyG:        {torch_geometric.__version__}")
except Exception as exc:
    print(f"PyG import failed: {exc}")

ram = psutil.virtual_memory()
print(
    f"CPU RAM:    total={ram.total/1024**3:.1f} GB, "
    f"available={ram.available/1024**3:.1f} GB"
)

for label, path in (
    ("workspace", WORKSPACE_ROOT),
    ("mstc_root", MSTC_ROOT),
    ("artifact_root", ARTIFACT_ROOT),
    ("data_root", DATA_ROOT),
):
    if path.exists():
        usage = shutil.disk_usage(path)
        print(
            f"{label}: total={usage.total/1024**3:.1f} GB, "
            f"free={usage.free/1024**3:.1f} GB"
        )
    else:
        print(f"{label}: not found ({path})")

if SOURCE_ARCHIVE.is_file():
    print(
        f"Source archive size: "
        f"{SOURCE_ARCHIVE.stat().st_size / 1024**3:.2f} GB "
        f"({SOURCE_ARCHIVE})"
    )
else:
    print(f"Source archive missing: {SOURCE_ARCHIVE}")

if GROUND_TRUTH_ROOT.exists():
    n_files = sum(
        1 for _ in GROUND_TRUTH_ROOT.rglob("*") if _.is_file()
    )
    print(f"Ground Truth files (recursive): {n_files}")
else:
    print(f"Ground Truth not yet present: {GROUND_TRUTH_ROOT}")

GPU_READY = torch.cuda.is_available()
print(f"GPU_READY: {GPU_READY}")


## 4. Install / Verify Python Dependencies

The pipeline already ran the heavy 13GB preprocessing in the original
Google Drive; the artifacts live in `/mnt/workspace/mstc_pids/artifacts`.
We do **not** install/upgrade `torch` itself — we only install
missing pure-Python dependencies and the optional PyG compiled
extensions that match the currently installed `torch` + CUDA.

In [ ]:
import subprocess
import sys


def _pip_show(name: str) -> bool:
    """Return True when `pip show` reports the package as installed."""
    return subprocess.run(
        [sys.executable, "-m", "pip", "show", name],
        capture_output=True, text=True,
    ).returncode == 0


if not INSTALL_DEPENDENCIES:
    print("INSTALL_DEPENDENCIES=False; skipping pip install.")
else:
    missing = [
        pkg for pkg in (
            "scikit-learn", "networkx", "xxhash", "graphviz", "psutil",
            "matplotlib", "wandb", "chardet", "nltk", "igraph",
            "cairocffi", "wget", "gensim", "pytz", "pandas", "yacs",
            "psycopg2-binary", "tqdm", "pyyaml", "torch_geometric",
        )
        if not _pip_show(pkg)
    ]
    if missing:
        print(f"Installing missing packages: {missing}")
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", *missing],
            check=True,
        )
    else:
        print("All tracked pure-Python packages already installed.")

    # Optional PyG compiled extensions — pinned to the current torch + CUDA.
    torch_version = torch.__version__.split("+")[0]
    cuda_tag = (
        "cpu" if torch.version.cuda is None
        else "cu" + torch.version.cuda.replace(".", "")
    )
    wheel_index = (
        f"https://data.pyg.org/whl/torch-{torch_version}+{cuda_tag}.html"
    )
    print(f"PyG wheel index: {wheel_index}")

    for pkg in ("pyg_lib", "torch_scatter", "torch_sparse"):
        try:
            __import__(pkg)
            print(f"{pkg}: already importable")
        except Exception:
            if not _pip_show(pkg):
                try:
                    subprocess.run(
                        [
                            sys.executable, "-m", "pip", "install",
                            "--quiet", pkg, "-f", wheel_index,
                        ],
                        check=True,
                    )
                    print(f"{pkg}: installed")
                except subprocess.CalledProcessError as exc:
                    raise RuntimeError(
                        f"Failed to install {pkg}. "
                        f"Python={sys.version.split()[0]}, "
                        f"torch={torch.__version__}, "
                        f"CUDA={torch.version.cuda}, "
                        f"wheel_index={wheel_index}, "
                        f"pip rc={exc.returncode}."
                    ) from exc
            else:
                print(f"{pkg}: pip-show reports installed but import failed; "
                      "investigate manually before proceeding.")


## 5. Import Smoke Test and Environment Record

Smoke-import the core stack and persist a pip-freeze snapshot and an
`environment/pai_environment.json` record describing the runtime.

In [ ]:
import importlib
import subprocess
import sys
import json
from datetime import datetime, timezone

for module_name in (
    "torch", "torch_geometric", "pandas", "yaml", "gensim", "nltk",
    "networkx", "psutil", "sklearn",
):
    try:
        module = importlib.import_module(module_name)
        ver = getattr(module, "__version__", "")
        print(f"import {module_name}: ok ({ver})")
    except Exception as exc:
        print(f"import {module_name}: FAILED ({exc})")

# Pip freeze for environment record.
freeze_path = ENVIRONMENT_DIR / "pai_pip_freeze.txt"
with freeze_path.open("w", encoding="utf-8") as fh:
    subprocess.run(
        [sys.executable, "-m", "pip", "freeze"],
        stdout=fh, check=True, text=True,
    )
print(f"Environment freeze: {freeze_path}")


In [ ]:
env_record = {
    "platform": "aliyun_pai_dsw",
    "workspace_root": str(WORKSPACE_ROOT),
    "project_root": str(PROJECT_ROOT),
    "mstc_root": str(MSTC_ROOT),
    "artifact_root": str(ARTIFACT_ROOT),
    "data_root": str(DATA_ROOT),
    "source_archive": str(SOURCE_ARCHIVE),
    "source_archive_sha256": SOURCE_ARCHIVE_SHA256,
    "source_kind": SOURCE_KIND,
    "repository_ref": REPOSITORY_REF,
    "expected_production_commit": EXPECTED_PRODUCTION_COMMIT,
    "ground_truth_commit": GROUND_TRUTH_COMMIT,
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "torch_geometric_version": getattr(
        importlib.import_module("torch_geometric"), "__version__", None
    ),
    "cuda_available": bool(torch.cuda.is_available()),
    "cuda_version": torch.version.cuda,
    "gpu_name": (
        torch.cuda.get_device_name(0)
        if torch.cuda.is_available() else None
    ),
    "git_commit": None,
    "timestamp": datetime.now(timezone.utc).isoformat(),
}

env_path = ENVIRONMENT_DIR / "pai_environment.json"
with env_path.open("w", encoding="utf-8") as fh:
    json.dump(env_record, fh, indent=2, ensure_ascii=False)
print(f"Environment record: {env_path}")
print(f"platform=aliyun_pai_dsw; git_commit=null (GitHub ZIP)")


## 6. Resolve Configuration and Inspect Persistent Artifacts

Reuse the production helper semantics to inspect preprocessing
artifacts. When `build_graphs`, `metadata`, `embed_nodes`,
`embed_edges` are all complete, the pipeline can run in
`detection_only` mode — no PostgreSQL restore needed.

In [ ]:
from config import get_runtime_required_args, get_yml_cfg
from pipeline_stages import (
    check_preprocess_stage_complete,
    check_all_preprocess_stages_complete,
)

PREPROCESS_CONFIG = PROJECT_ROOT / "config" / "orthrus.yml"


def fresh_preprocess_cfg():
    args = get_runtime_required_args(args=[
        DATASET, "--config", str(PREPROCESS_CONFIG),
        "--artifact-root", str(ARTIFACT_ROOT), "--stages", "preprocess",
        f"--semantic_features.corpus_scope={CORPUS_SCOPE}",
        "--skip-tracing",
    ])
    return get_yml_cfg(args)


def artifact_stats(path):
    """Count + size of visible files under `path` (recursive)."""
    path = Path(path)
    if not path.is_dir():
        return {"files": 0, "MB": 0.0, "GB": 0.0}
    files = [p for p in path.rglob("*") if p.is_file()]
    total = sum(p.stat().st_size for p in files)
    return {
        "files": len(files),
        "MB": total / 1024**2,
        "GB": total / 1024**3,
    }


def read_preprocess_status(cfg):
    status = {
        stage: check_preprocess_stage_complete(cfg, stage)
        for stage in (
            "build_graphs", "embed_nodes", "embed_edges", "metadata",
        )
    }
    status["artifacts_complete"] = (
        check_all_preprocess_stages_complete(cfg)
    )
    return status


def print_status(label, status):
    print(label)
    for stage in (
        "build_graphs", "embed_nodes", "embed_edges", "metadata",
    ):
        print(f"  {stage}: {'Y' if status[stage] else 'N'}")
    print(f"  artifacts_complete: {status['artifacts_complete']}")


cfg = fresh_preprocess_cfg()
status_before = read_preprocess_status(cfg)
print_status("Current:", status_before)

artifact_paths = {
    "build_graphs": cfg.graph_construction.build_graphs._graphs_dir,
    "word2vec": cfg.edge_featurization.embed_nodes.feature_word2vec._model_dir,
    "edge_embeddings": cfg.edge_featurization.embed_edges._edge_embeds_dir,
    "metadata": cfg._metadata_dir,
    "whole_ARTIFACT_ROOT": ARTIFACT_ROOT,
}
for label, p in artifact_paths.items():
    stats = artifact_stats(p)
    print(
        f"{label}: files={stats['files']}, "
        f"MB={stats['MB']:.2f}, GB={stats['GB']:.3f}"
    )

build_graphs = status_before["build_graphs"]
metadata = status_before["metadata"]
embed_nodes = status_before["embed_nodes"]
embed_edges = status_before["embed_edges"]
ALL_COMPLETE = status_before["artifacts_complete"]

DATABASE_REQUIRED_FOR_RESUME = (
    not build_graphs or not metadata
)
PREPROCESS_REQUIRED = not ALL_COMPLETE

if ALL_COMPLETE:
    print("\nDecision: ALL COMPLETE.")
    print("  - No database restore needed.")
    print("  - No preprocessing needed.")
    print("  - Next: GPU Baseline Smoke Test.")
elif DATABASE_REQUIRED_FOR_RESUME:
    print("\nDecision: Database restore required.")
    print("  - build_graphs or metadata incomplete.")
    print("  - Configure ORTHRUS_DB_* env vars + RUN_DB_PREFLIGHT=True.")
elif not embed_nodes or not embed_edges:
    print("\nDecision: Restartable preprocessing (no DB restore).")
    print("  - build_graphs + metadata complete.")
    print("  - Set RUN_PREPROCESS=True to resume embed_nodes/embed_edges.")
else:
    print("\nDecision: Status unclear; manual inspection required.")


## 7. Optional Database Preflight

PAI-DSW never auto-installs PostgreSQL. If artifacts are complete
we have zero database dependencies. When build_graphs or metadata
is missing, the user is responsible for providing an external
PostgreSQL endpoint via the `ORTHRUS_DB_*` environment variables.

In [ ]:
if not RUN_DB_PREFLIGHT:
    print("RUN_DB_PREFLIGHT=False; PostgreSQL untouched.")
else:
    import os
    import psycopg2

    user = os.environ.get("ORTHRUS_DB_USER") or ORTHRUS_DB_USER
    host = os.environ.get("ORTHRUS_DB_HOST") or ORTHRUS_DB_HOST
    port = os.environ.get("ORTHRUS_DB_PORT") or ORTHRUS_DB_PORT
    password = (
        os.environ.get("ORTHRUS_DB_PASSWORD") or ORTHRUS_DB_PASSWORD
    )
    if not (user and host and port and password):
        raise RuntimeError(
            "RUN_DB_PREFLIGHT=True but one or more of "
            "ORTHRUS_DB_HOST / PORT / USER / PASSWORD is empty."
        )
    connection = psycopg2.connect(
        user=user, host=host, port=int(port),
        password=password, dbname="postgres",
    )
    cursor = connection.cursor()
    try:
        cursor.execute("SELECT 1;")
        print(f"ORTHRUS database preflight OK at {host}:{port}")
    finally:
        cursor.close()
        connection.close()


## 8. Optional Bounded-Memory Benchmark

Preprocessing memory benchmark. Same script as Colab, but PAI
runs in CPU mode. Default skipped.

In [ ]:
import subprocess
import sys

if not RUN_PREPROCESS_BENCHMARK:
    print("RUN_PREPROCESS_BENCHMARK=False; benchmark skipped.")
else:
    benchmark_cmd = [
        sys.executable,
        str(PROJECT_ROOT / "scripts" / "benchmark_preprocess_memory.py"),
        "--events", "200000", "--fetch-size", "8192",
    ]
    result = subprocess.run(
        benchmark_cmd, cwd=PROJECT_ROOT, check=False,
    )
    if result.returncode:
        raise RuntimeError(
            "Preprocessing benchmark failed; formal preprocessing blocked."
        )


## 9. Restartable Preprocessing

Restartable preprocessing. Uses `--cpu` because preprocessing has
no GPU benefit. Streamed output, exit-code check, completion-marker
verification, persistent log.

In [ ]:
import os
import subprocess
import sys


def run_streamed_command(command, *, cwd, log_path, env=None):
    """Run a subprocess with streamed stdout/stderr to notebook + log.

    - stdout + stderr are merged.
    - PYTHONUNBUFFERED=1 is forced.
    - shell=False (the caller passes a list).
    - Non-zero return code → RuntimeError.
    """
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    child_env = dict(os.environ if env is None else env)
    child_env["PYTHONUNBUFFERED"] = "1"
    log_fh = log_path.open("w", encoding="utf-8")
    try:
        proc = subprocess.Popen(
            command, cwd=cwd,
            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
            text=True, bufsize=1, env=child_env,
        )
        assert proc.stdout is not None
        for line in proc.stdout:
            print(line, end="")
            log_fh.write(line)
            log_fh.flush()
        proc.wait()
        return proc.returncode
    finally:
        log_fh.close()


if not RUN_PREPROCESS:
    print("RUN_PREPROCESS=False; preprocessing skipped.")
else:
    cfg = fresh_preprocess_cfg()
    before = read_preprocess_status(cfg)
    print_status("Before:", before)

    if not before["build_graphs"] or not before["metadata"]:
        env_db_keys = [k for k in os.environ if k.startswith("ORTHRUS_DB_")]
        if not env_db_keys:
            raise RuntimeError(
                "build_graphs or metadata incomplete; configure ORTHRUS_DB_* "
                "before preprocessing."
            )

    preprocess_cmd = [
        sys.executable, str(PROJECT_ROOT / "src" / "orthrus.py"),
        DATASET,
        "--config", str(PREPROCESS_CONFIG),
        "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "preprocess",
        "--preprocess-substages", PREPROCESS_SUBSTAGES,
        f"--semantic_features.corpus_scope={CORPUS_SCOPE}",
        "--skip-tracing",
        "--cpu",
    ]
    if FORCE_PREPROCESS:
        preprocess_cmd.append("--force-preprocess")

    print(f"Command: {' '.join(preprocess_cmd)}")
    rc = run_streamed_command(
        preprocess_cmd,
        cwd=PROJECT_ROOT,
        log_path=ENVIRONMENT_DIR / "pai_preprocess_latest.log",
    )
    print(f"Preprocessing exited with code: {rc}")
    if rc != 0:
        raise RuntimeError(
            f"Preprocessing failed (rc={rc}). See "
            f"{ENVIRONMENT_DIR / 'pai_preprocess_latest.log'}."
        )

cfg = fresh_preprocess_cfg()
after = read_preprocess_status(cfg)
print_status("After:", after)

if not after["artifacts_complete"]:
    incomplete = [
        s for s in (
            "build_graphs", "metadata", "embed_nodes", "embed_edges",
        )
        if not after[s]
    ]
    raise RuntimeError(
        f"RUN_PREPROCESS=True but artifacts still incomplete: {incomplete}."
    )
print("\nTHEIA preprocessing COMPLETE")


## 10. Baseline Smoke Test + Loader Telemetry Verification

The paper defines a smoke test as 1 epoch on `baseline.yml`, with
tracing disabled and `--max-windows-per-split 2`. We additionally
verify C8 loader telemetry persistence on the smoke output.

In [ ]:
import json
import shutil
import sys
from pathlib import Path

import yaml


def _runtime_json_with_loader_telemetry(run_dir: Path) -> Path:
    """Locate the smoke runtime.json.

    Bounded smoke isolates artifacts under
    <artifact_root>/<dataset>/runs/<model>/smoke/seed_<n>.
    We scan that subtree for any runtime.json that carries the
    dataset_loader -> training|testing telemetry markers.
    """
    candidates: list[Path] = []
    for path in run_dir.rglob("runtime.json"):
        try:
            payload = json.loads(path.read_text(encoding="utf-8"))
        except Exception:
            continue
        loader = payload.get("dataset_loader")
        if isinstance(loader, dict) and (
            "training" in loader or "testing" in loader
        ):
            candidates.append(path)
    if not candidates:
        raise FileNotFoundError(
            f"No runtime.json with dataset_loader telemetry under {run_dir}"
        )
    candidates.sort(key=lambda p: p.stat().st_mtime, reverse=True)
    return candidates[0]


def assert_loader_telemetry_persistence(artifact_root: Path) -> Path:
    """C8 loader-telemetry persistence acceptance.

    Sections required:
      - dataset_loader.training
      - dataset_loader.testing
      - training, testing, model (any nested detail block)
    """
    runtime_path = _runtime_json_with_loader_telemetry(artifact_root)
    runtime = json.loads(runtime_path.read_text(encoding="utf-8"))

    loader = runtime.get("dataset_loader")
    if not isinstance(loader, dict):
        raise AssertionError(
            f"runtime.json missing dataset_loader section: {runtime_path}"
        )
    if not isinstance(loader.get("training"), dict):
        raise AssertionError(
            f"dataset_loader.training missing: {runtime_path}"
        )
    if not isinstance(loader.get("testing"), dict):
        raise AssertionError(
            f"dataset_loader.testing missing: {runtime_path}"
        )
    for section in ("training", "testing", "model"):
        if not isinstance(runtime.get(section), dict):
            raise AssertionError(
                f"runtime.{section} missing: {runtime_path}"
            )

    print("=" * 40)
    print("LOADER TELEMETRY PERSISTENCE: PASSED")
    print("dataset_loader.training: PRESENT")
    print("dataset_loader.testing:  PRESENT")
    print("training/testing/model:  PRESENT")
    print("=" * 40)
    return runtime_path


def write_smoke_config() -> Path:
    """Render the bounded-smoke YAML.

    Same semantics as the Colab Notebook:
      - detection.gnn_training.num_epochs = 1
      - pipeline.run_tracing = False
    """
    base_config = PROJECT_ROOT / "config" / "experiments" / "baseline.yml"
    payload = yaml.safe_load(base_config.read_text(encoding="utf-8"))
    payload.setdefault("pipeline", {})["run_tracing"] = False
    payload.setdefault("detection", {}).setdefault(
        "gnn_training", {}
    )["num_epochs"] = 1
    out = ARTIFACT_ROOT / "environment" / "pai_smoke_baseline_1epoch.yml"
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(
        yaml.safe_dump(payload, sort_keys=False),
        encoding="utf-8",
    )
    return out


In [ ]:
if not RUN_BASELINE_SMOKE:
    print("RUN_BASELINE_SMOKE=False; bounded smoke skipped.")
elif not GPU_READY:
    print("GPU not available; smoke must NOT run on CPU.")
else:
    SMOKE_CONFIG = write_smoke_config()
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_experiment.py"),
        "--dataset", DATASET,
        "--config", str(SMOKE_CONFIG),
        "--seed", str(SEEDS[0]),
        "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "train,test,evaluate",
        "--max-windows-per-split", str(SMOKE_MAX_WINDOWS_PER_SPLIT),
    ]
    print("=" * 60)
    print("Baseline Smoke Test")
    print("=" * 60)
    print(f"Dataset: {DATASET}, Seed: {SEEDS[0]}, Epochs: 1")
    print(f"Max windows/split: {SMOKE_MAX_WINDOWS_PER_SPLIT}")
    print(f"Config:  {SMOKE_CONFIG}")
    print(f"Command: {' '.join(cmd)}")

    rc = run_streamed_command(
        cmd,
        cwd=PROJECT_ROOT,
        log_path=ENVIRONMENT_DIR / "pai_baseline_smoke_latest.log",
    )
    if rc != 0:
        raise RuntimeError(
            f"Baseline smoke failed (rc={rc}). See "
            f"{ENVIRONMENT_DIR / 'pai_baseline_smoke_latest.log'}."
        )
    print("=" * 60)
    print("Baseline bounded smoke PASSED")

    # C8 loader-telemetry acceptance (runs only when smoke succeeded).
    assert_loader_telemetry_persistence(ARTIFACT_ROOT)


## 11. Main Model Matrix

Main model matrix: `baseline.yml` + `mstc_full.yml` on THEIA_E3
(seed 0). No `--max-windows-per-split`. `--force` is **only** added
when the user explicitly opts in via `FORCE_MAIN_MATRIX=True`.

Streamed output goes to `pai_main_matrix_latest.log`; the
`run_matrix.py` scheduler owns `completed/skipped/failed` and
stale-running recovery. We do not re-implement any of that here.

In [ ]:
import subprocess
import sys

if not RUN_MAIN_MATRIX:
    print("RUN_MAIN_MATRIX=False; main model matrix skipped.")
elif not GPU_READY:
    print("GPU not available; main matrix must NOT run on CPU.")
else:
    MAIN_CONFIGS = [
        PROJECT_ROOT / "config" / "experiments" / "baseline.yml",
        PROJECT_ROOT / "config" / "experiments" / "mstc_full.yml",
    ]
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_matrix.py"),
        "--datasets", DATASET,
        "--configs", ",".join(map(str, MAIN_CONFIGS)),
        "--seeds", ",".join(map(str, SEEDS)),
        "--artifact-root", str(ARTIFACT_ROOT),
    ]
    if FORCE_MAIN_MATRIX:
        cmd.append("--force")

    print("=" * 60)
    print("Main Model Matrix")
    print("=" * 60)
    print(f"Configs: {[c.name for c in MAIN_CONFIGS]}")
    print(f"Datasets: {DATASET}; Seeds: {SEEDS}")
    print(f"Command: {' '.join(cmd)}")
    print()
    print("Monitor processes (live):")
    try:
        subprocess.run(
            ["ps", "-eo", "pid,etime,%cpu,%mem,cmd"],
            check=False,
        )
    except Exception:
        pass
    print(f"run_status markers: {ARTIFACT_ROOT / 'results' / 'run_status'}")
    print()

    rc = run_streamed_command(
        cmd,
        cwd=PROJECT_ROOT,
        log_path=ENVIRONMENT_DIR / "pai_main_matrix_latest.log",
    )
    if rc != 0:
        raise RuntimeError(
            f"Main matrix failed (rc={rc}). See "
            f"{ENVIRONMENT_DIR / 'pai_main_matrix_latest.log'}."
        )
    print("=" * 60)
    print("Main model matrix complete.")


## 12. Ablations and Specialized Experiments

All ablation/specialized groups from the Colab Notebook, routed
through `run_matrix.py`. Same config sets, no semantic changes.

In [ ]:
import sys

GROUPS = {
    "ablation": [
        "baseline.yml", "ablation_no_multiscale.yml",
        "ablation_no_gate.yml", "ablation_no_time.yml",
        "ablation_no_calibration.yml", "ablation_no_topk.yml",
        "mstc_full.yml",
    ],
    "multiscale": [
        "multiscale_recent20.yml", "multiscale_recent24.yml",
        "multiscale_single_window.yml", "multiscale_equal.yml",
        "multiscale_gate.yml",
    ],
    "time": [
        "time_type_only.yml", "time_time_only.yml", "time_joint.yml",
    ],
    "calibration": [
        "calibration_max.yml", "calibration_quantile.yml",
        "calibration_kmeans.yml", "calibration_global_p.yml",
        "calibration_relation.yml", "calibration_hierarchical.yml",
    ],
    "backbone": [
        "backbone_graphtransformer.yml",
        "backbone_graphsage_baseline.yml",
        "backbone_graphsage.yml", "backbone_mlp.yml",
    ],
    "dataset_view": [
        "host_only.yml", "host_network_structure.yml",
        "host_network_full.yml",
    ],
    "efficiency": [
        "baseline.yml", "efficiency_multiscale.yml",
        "efficiency_multiscale_time.yml", "mstc_full.yml",
    ],
}

if not RUN_ABLATIONS:
    print("RUN_ABLATIONS=False; ablations skipped.")
elif not GPU_READY:
    print("GPU not available; ablations must NOT run on CPU.")
else:
    if EXPERIMENT_GROUP not in GROUPS:
        raise ValueError(
            f"Unknown EXPERIMENT_GROUP {EXPERIMENT_GROUP!r}; "
            f"available: {sorted(GROUPS)}"
        )
    cfg_paths = [
        PROJECT_ROOT / "config" / "experiments" / name
        for name in GROUPS[EXPERIMENT_GROUP]
    ]
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_matrix.py"),
        "--datasets", DATASET,
        "--configs", ",".join(map(str, cfg_paths)),
        "--seeds", ",".join(map(str, SEEDS)),
        "--artifact-root", str(ARTIFACT_ROOT),
    ]
    if FORCE_ABLATIONS:
        cmd.append("--force")

    print("=" * 60)
    print(f"Ablations: {EXPERIMENT_GROUP}")
    print("=" * 60)
    print(f"Configs: {GROUPS[EXPERIMENT_GROUP]}")
    print(f"Command: {' '.join(cmd)}")
    rc = run_streamed_command(
        cmd,
        cwd=PROJECT_ROOT,
        log_path=ENVIRONMENT_DIR / "pai_ablations_latest.log",
    )
    if rc != 0:
        raise RuntimeError(
            f"Ablations failed (rc={rc}). See "
            f"{ENVIRONMENT_DIR / 'pai_ablations_latest.log'}."
        )
    print("Ablation experiments complete.")


## 13. Checkpoint Resume

Manual checkpoint resume. The user supplies `CHECKPOINT`; we only
run when `RUN_MANUAL_RESUME=True`. Missing checkpoint → FileNotFoundError.

In [ ]:
import subprocess
import sys
from pathlib import Path

if not RUN_MANUAL_RESUME:
    print("RUN_MANUAL_RESUME=False; no checkpoint loaded.")
elif not GPU_READY:
    print("GPU not available; resume must NOT run on CPU.")
else:
    if not CHECKPOINT or not Path(CHECKPOINT).exists():
        raise FileNotFoundError(
            f"Checkpoint does not exist: {CHECKPOINT}"
        )
    RESUME_CONFIG = PROJECT_ROOT / RESUME_CONFIG_REL
    cmd = [
        sys.executable,
        str(PROJECT_ROOT / "src" / "experiments" / "run_experiment.py"),
        "--dataset", DATASET,
        "--config", str(RESUME_CONFIG),
        "--seed", str(SEEDS[0]),
        "--artifact-root", str(ARTIFACT_ROOT),
        "--stages", "train,test,evaluate",
        "--checkpoint", str(CHECKPOINT),
    ]
    print("=" * 60)
    print("Checkpoint Resume")
    print("=" * 60)
    print(f"Config:    {RESUME_CONFIG}")
    print(f"Checkpoint:{CHECKPOINT}")
    print(f"Command: {' '.join(cmd)}")
    rc = run_streamed_command(
        cmd, cwd=PROJECT_ROOT,
        log_path=ENVIRONMENT_DIR / "pai_resume_latest.log",
    )
    if rc != 0:
        raise RuntimeError(f"Resume failed (rc={rc}).")
    print("Resume complete.")


## 14. Result Collection and Export

Collect results + export tables. The two helpers are the same
production scripts used by the Colab Notebook; semantics preserved.

In [ ]:
import subprocess
import sys

if not RUN_COLLECT_EXPORT:
    print("RUN_COLLECT_EXPORT=False; result collection/export skipped.")
else:
    print("=" * 60)
    print("Result Collection and Export")
    print("=" * 60)
    for script_name in ("collect_results.py", "export_tables.py"):
        cmd = [
            sys.executable,
            str(PROJECT_ROOT / "src" / "experiments" / script_name),
            "--artifact-root", str(ARTIFACT_ROOT),
        ]
        rc = run_streamed_command(
            cmd, cwd=PROJECT_ROOT,
            log_path=(
                ENVIRONMENT_DIR /
                f"pai_{script_name.replace('.py', '')}_latest.log"
            ),
        )
        if rc != 0:
            raise RuntimeError(
                f"{script_name} failed (rc={rc})."
            )
    print("Result collection and export complete.")


## 15. Result Display

Display the standard result tables. Missing tables produce a
warning rather than an error.

In [ ]:
import pandas as pd
from pathlib import Path

if not RUN_DISPLAY_RESULTS:
    print("RUN_DISPLAY_RESULTS=False; display skipped.")
else:
    print("=" * 60)
    print("Result Display")
    print("=" * 60)
    RESULTS_ROOT = ARTIFACT_ROOT / "results"
    tables = [
        "all_runs.csv",
        "main_results.csv",
        "ablation_results.csv",
        "calibration_results.csv",
        "efficiency_results.csv",
    ]
    for name in tables:
        path = RESULTS_ROOT / name
        if not path.is_file():
            print(f"Warning: {name} missing")
            continue
        df = pd.read_csv(path)
        print(f"\n--- {name} ({len(df)} rows) ---")
        try:
            from IPython.display import display
            display(df)
        except Exception:
            print(df.head().to_string())
    print("=" * 60)
